In [8]:
%load_ext autoreload
%autoreload 2

In [9]:
import os
from pathlib import Path

if "notebooks" in os.getcwd():
    os.chdir(Path.cwd().parent)

In [10]:
import logging

logger = logging.getLogger("ts_visualization")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(levelname)s: %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)


In [11]:
from xaitimesynth.generators import TimeSeriesGenerator
import numpy as np
import pandas as pd
import logging
from lets_plot import *

LetsPlot.setup_html()

TimeSeriesGenerator()


# Helper functions

In [12]:
def theme_presentation(
    legend_position="bottom", legend_key_height=None, legend_key_width=None
):
    from lets_plot import theme, element_text

    return theme(
        # Title and subtitle
        plot_title=element_text(family="Arial", size=24),
        plot_subtitle=element_text(family="Arial", size=20),
        # subtitle=element_text(family="Arial", size=20),
        # Axis titles
        axis_title=element_text(family="Arial", size=20),
        # Axis text (tick labels)
        axis_text=element_text(family="Arial", size=18),
        # Legend
        legend_title=element_text(family="Arial", size=20),
        legend_text=element_text(family="Arial", size=18),
        # Additional styling for clarity
        legend_position=legend_position,
        legend_key_height=legend_key_height,
        legend_key_width=legend_key_width,
    )


# Synth Data

In [13]:
def extract_feature_masks(dataset):
    """Extract feature masks from dataset."""
    # Extract all feature masks
    feature_masks = {}

    # First, try the feature_masks key directly if it exists
    if "feature_masks" in dataset:
        if isinstance(dataset["feature_masks"], dict):
            logger.debug("Found feature_masks as dictionary")
            for key, mask in dataset["feature_masks"].items():
                feature_masks[key] = mask
        elif isinstance(dataset["feature_masks"], np.ndarray):
            logger.debug("Found feature_masks as array")
            feature_masks["feature_masks"] = dataset["feature_masks"]

    # Look for feature mask keys with "feature_masks" in the name
    for key in dataset.keys():
        if (
            isinstance(dataset[key], np.ndarray)
            and "feature_masks" in key
            and key != "feature_masks"
        ):
            feature_masks[key] = dataset[key]
            logger.debug(f"Found feature mask: {key}, shape: {dataset[key].shape}")

    if not feature_masks:
        logger.warning("No feature masks found in dataset")
    else:
        mask_count = len(feature_masks)
        logger.info(f"Found {mask_count} feature masks in dataset")

    return feature_masks


def create_rectangle_data(dataset, feature_masks):
    """Create rectangle data from extracted feature masks."""
    rectangles = []

    # Get sample indices for each class
    class0_idx = np.where(dataset["y"] == 0)[0][0]
    class1_idx = np.where(dataset["y"] == 1)[0][0]

    # Process each class
    for class_idx, idx in [(0, class0_idx), (1, class1_idx)]:
        logger.debug(f"Processing Class {class_idx}")

        # Find relevant masks for this class
        class_masks = {}
        for key, mask in feature_masks.items():
            # Try to extract class index from the key if possible
            if f"class_{class_idx}" in key.lower():
                if len(mask.shape) >= 2:
                    class_masks[key] = mask[idx]
                else:
                    class_masks[key] = mask

        # If no class-specific masks were found, check all masks
        if not class_masks and class_idx == 1:  # We expect Class 1 to have features
            logger.debug("No class-specific masks found, checking all masks")

            # Manually check each mask
            for key, mask in feature_masks.items():
                if (
                    len(mask.shape) >= 2
                    and mask.shape[0] >= max(class0_idx, class1_idx) + 1
                ):
                    sample_mask = mask[idx]
                    if np.any(sample_mask):
                        logger.debug(
                            f"Using mask {key} with {np.sum(sample_mask)} True values"
                        )
                        class_masks[key] = sample_mask

        # For each mask, find contiguous regions
        regions_found = False
        for mask_key, mask_array in class_masks.items():
            if not np.any(mask_array):
                continue

            # Find contiguous regions
            changes = np.diff(
                np.concatenate([[False], mask_array, [False]]).astype(int)
            )
            start_indices = np.where(changes == 1)[0]
            end_indices = np.where(changes == -1)[0]

            if len(start_indices) > 0:
                regions_found = True

            # Create rectangle for each region
            for start, end in zip(start_indices, end_indices):
                logger.debug(f"Adding rectangle: Class {class_idx}, {start}-{end}")
                rectangles.append(
                    {
                        "class": f"Class {class_idx}",
                        "component": "Full Series",  # Only show in Full Series panels
                        "xmin": float(start),
                        "xmax": float(end),
                    }
                )

        # If we expected to find features but didn't, log a warning
        if class_idx == 1 and not regions_found:
            logger.warning(f"No feature regions found for Class {class_idx}")

            # Check if we can extract feature regions from components directly
            if "components" in dataset:
                logger.info("Attempting to extract feature regions from components")

                comp = dataset["components"][idx]
                if hasattr(comp, "feature_masks") and comp.feature_masks:
                    logger.debug("Found feature masks in component")

                    for feature_name, feature_mask in comp.feature_masks.items():
                        if np.any(feature_mask):
                            changes = np.diff(
                                np.concatenate([[False], feature_mask, [False]]).astype(
                                    int
                                )
                            )
                            start_indices = np.where(changes == 1)[0]
                            end_indices = np.where(changes == -1)[0]

                            for start, end in zip(start_indices, end_indices):
                                logger.debug(
                                    f"Adding component rectangle: {start}-{end}"
                                )
                                rectangles.append(
                                    {
                                        "class": f"Class {class_idx}",
                                        "component": "Full Series",
                                        "xmin": float(start),
                                        "xmax": float(end),
                                    }
                                )

    # If we still don't have rectangles for Class 1, use the features directly
    if not any(r["class"] == "Class 1" for r in rectangles):
        logger.warning("Using features directly to determine feature regions")

        if "components" in dataset:
            comp = dataset["components"][class1_idx]
            if hasattr(comp, "features") and comp.features:
                for feature_name, feature_values in comp.features.items():
                    # Find where the feature has non-zero values
                    non_zero = np.abs(feature_values) > 1e-6
                    if np.any(non_zero):
                        changes = np.diff(
                            np.concatenate([[False], non_zero, [False]]).astype(int)
                        )
                        start_indices = np.where(changes == 1)[0]
                        end_indices = np.where(changes == -1)[0]

                        for start, end in zip(start_indices, end_indices):
                            logger.debug(
                                f"Adding feature region from values: {start}-{end}"
                            )
                            rectangles.append(
                                {
                                    "class": "Class 1",
                                    "component": "Full Series",
                                    "xmin": float(start),
                                    "xmax": float(end),
                                }
                            )

    if rectangles:
        logger.info(f"Created {len(rectangles)} feature indicator rectangles")
    else:
        logger.warning("No feature indicator rectangles were created")

    return pd.DataFrame(rectangles) if rectangles else None


def create_visualization_data(dataset):
    """Create data for visualization."""
    logger.info("Preparing time series data for visualization")

    # Get sample indices for each class
    class0_idx = np.where(dataset["y"] == 0)[0][0]
    class1_idx = np.where(dataset["y"] == 1)[0][0]

    # Get components
    comp0 = dataset["components"][class0_idx]
    comp1 = dataset["components"][class1_idx]

    # Create main time series data
    data_rows = []

    # Define components to include and their order
    components = ["Full Series", "Features", "Base Structure", "Noise"]

    # Time steps
    n_timesteps = len(dataset["X"][class0_idx])

    # Process each class
    for class_idx, comp, idx in [(0, comp0, class0_idx), (1, comp1, class1_idx)]:
        # Full Series
        for t, val in enumerate(dataset["X"][idx]):
            data_rows.append(
                {
                    "time": float(t),
                    "value": float(val),
                    "class": f"Class {class_idx}",
                    "component": "Full Series",
                }
            )

        # Features
        if not comp.features:
            # Empty features for Class 0
            for t in range(n_timesteps):
                data_rows.append(
                    {
                        "time": float(t),
                        "value": 0.0,
                        "class": f"Class {class_idx}",
                        "component": "Features",
                    }
                )
        else:
            # Combine all features
            combined_features = np.zeros(n_timesteps)
            for name, feature in comp.features.items():
                combined_features += feature

            for t, val in enumerate(combined_features):
                data_rows.append(
                    {
                        "time": float(t),
                        "value": float(val),
                        "class": f"Class {class_idx}",
                        "component": "Features",
                    }
                )

        # Base Structure
        for t, val in enumerate(comp.base_structure):
            data_rows.append(
                {
                    "time": float(t),
                    "value": float(val),
                    "class": f"Class {class_idx}",
                    "component": "Base Structure",
                }
            )

        # Noise
        for t, val in enumerate(comp.noise):
            data_rows.append(
                {
                    "time": float(t),
                    "value": float(val),
                    "class": f"Class {class_idx}",
                    "component": "Noise",
                }
            )

    # Create DataFrame
    df = pd.DataFrame(data_rows)

    # Sort components in the desired order
    df["component"] = pd.Categorical(
        df["component"], categories=components, ordered=True
    )

    logger.debug(f"Created visualization dataframe with shape: {df.shape}")
    return df


def create_ts_visualization(
    df, rectangles, show_indicators=True, facet_order={"x": "component", "y": "class"}
):
    """Create time series visualization with feature indicators as rectangles."""
    logger.info("Creating time series visualization with feature indicators")

    # Calculate global y-limits
    y_min = df["value"].min()
    y_max = df["value"].max()
    padding = (y_max - y_min) * 0.15
    y_min = float(y_min - padding)
    y_max = float(y_max + padding)

    # Start building the plot
    p = ggplot(df, aes(x="time", y="value"))

    # Add the rectangles for feature regions
    if show_indicators and rectangles is not None and not rectangles.empty:
        # Set rectangle height to full panel height
        rectangles = rectangles.copy()
        rectangles["ymin"] = y_min
        rectangles["ymax"] = y_max

        logger.debug(f"Adding {len(rectangles)} feature indicators to visualization")

        # Add rectangles to the plot
        p = p + geom_rect(
            data=rectangles,
            mapping=aes(xmin="xmin", xmax="xmax", ymin="ymin", ymax="ymax"),
            fill="red",
            alpha=0.25,
            size=0.15,
        )

    # Complete the plot
    p = (
        p
        + geom_hline(yintercept=0, linetype="dashed", color="gray")
        + geom_line(size=2)
        + facet_grid(**facet_order, x_order=0, y_order=0)
        + scale_y_continuous(limits=[y_min, y_max])
        + labs(x="Time", y="Value")
        + theme_bw()
    )

    return p


def generate_dataset_with_random_features(seed=None):
    """Generate a dataset with random feature locations."""
    logger.info("Generating synthetic dataset with random feature locations")

    generator = TimeSeriesGenerator(
        n_timesteps=100, n_samples=10, normalize=True, random_state=seed
    )

    # Class definitions with random locations
    class_definitions = [
        # Class 0: No discriminative feature
        {
            "base_structure": {"type": "random_walk", "step_size": 0.2},
            "noise": {"type": "gaussian", "sigma": 0.1},
            "features": [],
        },
        # Class 1: Shapelet and level change with random locations
        {
            "base_structure": {"type": "random_walk", "step_size": 0.05},
            "noise": {"type": "gaussian", "sigma": 0.1},
            "features": [
                {
                    "type": "shapelet",
                    "random_location": True,  # Random location
                    "length_pct": 0.2,  # 10% of length
                    "scale": 1.0,
                },
                {
                    "type": "level_change",
                    "random_location": True,  # Random location
                    "length_pct": 0.2,  # 20% of length
                    "amplitude": -0.5,
                },
            ],
        },
    ]

    dataset = generator.generate_dataset(
        class_definitions=class_definitions, return_components=True
    )

    logger.debug(f"Generated dataset with keys: {list(dataset.keys())}")

    # Log metadata to understand feature location structure
    if "metadata" in dataset:
        logger.debug(f"Dataset metadata: {dataset['metadata']}")

    return dataset


def main(show_indicators=True, use_random_locations=True, seed=None):
    """Generate dataset and create visualization."""
    # Generate dataset (with random or fixed feature locations)
    if use_random_locations:
        dataset = generate_dataset_with_random_features(seed=seed)
    else:
        # Use the original fixed-location generator
        generator = TimeSeriesGenerator(
            n_timesteps=100, n_samples=10, normalize=True, random_state=seed
        )
        class_definitions = [
            # Class 0: No discriminative feature
            {
                "base_structure": {"type": "random_walk", "step_size": 0.2},
                "noise": {"type": "gaussian", "sigma": 0.1},
                "features": [
                    {
                        "type": "shapelet",
                        "start_pct": 0.2,
                        "end_pct": 0.5,
                        "scale": 1.0,
                    },
                ],
            },
            # Class 1: Fixed features
            {
                "base_structure": {"type": "random_walk", "step_size": 0.05},
                "noise": {"type": "gaussian", "sigma": 0.1},
                "features": [
                    {
                        "type": "shapelet",
                        "start_pct": 0.2,
                        "end_pct": 0.5,
                        "scale": 1.0,
                    },
                    {
                        "type": "level_change",
                        "start_pct": 0.6,
                        "end_pct": 0.8,
                        "amplitude": -0.5,
                    },
                ],
            },
        ]
        dataset = generator.generate_dataset(
            class_definitions=class_definitions, return_components=True
        )

    # Extract feature masks
    feature_masks = extract_feature_masks(dataset)

    # Create rectangle data
    rectangles = (
        create_rectangle_data(dataset, feature_masks) if show_indicators else None
    )

    # Create visualization data
    df = create_visualization_data(dataset)

    # Create and display plot
    plot = create_ts_visualization(df, rectangles, show_indicators)

    location_type = "random" if use_random_locations else "fixed"
    logger.info(f"Visualization complete with {location_type} feature locations")

    ggsave(plot, "synthetic_ts_features.svg")

    return plot.show()


if __name__ == "__main__":
    # Uncomment to enable debug logging
    # logger.setLevel(logging.DEBUG)

    # Generate with random feature locations
    main(show_indicators=True, use_random_locations=False, seed=None)


INFO: Found 3 feature masks in dataset
INFO: Created 3 feature indicator rectangles
INFO: Preparing time series data for visualization
INFO: Creating time series visualization with feature indicators
INFO: Visualization complete with fixed feature locations
